# Notebook 01 — Model C / Proposal 1: Accessibility & Usability Scoring

**CS19 Capstone Project — Multi-Modal Data Fusion for Automated Accessibility & Usability Scoring**

## Scope
This notebook implements the full **Katie Model C3 / Proposal 1** scoring pipeline:

- **L1** — Per-participant, per-task ceiling + AT blocker / high caps (hard)
- **L2** — Accessibility stream: AT medium, comprehension caps, cohort-aware weights (L2.7), L2.6 floor
- **L3** — Usability stream: severity-based caps, met-expectations moderator (L3.7)
- **L1.5** — Confident-positive floor (applied after L2/L3)
- **Rollups** — Task, cohort, project

## Key design principles (Katie Model C3)
1. **Scores only go down** from the participant's stated survey score.
2. **Categorical failure rule**: blocked/severe AT friction → score = 0.
3. **Diminishing returns**: 3+ frictions of the same type trigger a stricter cap than a single instance.
4. **Confidence weighting (L1.4)**: frictions below 70% confidence do not trigger caps; above 70% the deduction scales with confidence.

## Outputs
All outputs are written to `outputs/` relative to this notebook's directory.

> **Note on Proposal 2**: This notebook covers only Proposal 1 (Model C scoring refinement).
> Proposal 2 (L3.8 time moderation) is applied in Notebook 02 and **only affects the L3 usability score**.
> Accessibility scores produced here are **not changed by Proposal 2**.

In [1]:
from pathlib import Path
import sys, os

BASE_DIR   = Path(".").resolve()
INPUT_DIR  = BASE_DIR
OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SURVEY_PATH   = INPUT_DIR / "A_Test_Raw_Survey_Scores_AAMIInsuranceWebsite-Website.xls"
INSIGHTS_PATH = INPUT_DIR / "AAMIwebsite-IndividualInsights-2026-03-10(1).xlsx"

for p in [SURVEY_PATH, INSIGHTS_PATH]:
    assert p.exists(), f"Missing input: {p}"

print("BASE_DIR  :", BASE_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("Survey    :", SURVEY_PATH.name, "— exists:", SURVEY_PATH.exists())
print("Insights  :", INSIGHTS_PATH.name, "— exists:", INSIGHTS_PATH.exists())

BASE_DIR  : /home/ubuntu/cs19_capstone
OUTPUT_DIR: /home/ubuntu/cs19_capstone/outputs
Survey    : A_Test_Raw_Survey_Scores_AAMIInsuranceWebsite-Website.xls — exists: True
Insights  : AAMIwebsite-IndividualInsights-2026-03-10(1).xlsx — exists: True


In [2]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer, util


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/usr/lib/python3.11/runpy.py", line 198, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/usr/lib/python3.11/runpy.py", line 88, in _run_code
    exec(code, run_globals)
  File "/usr/local/lib/python3.11/dist-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/usr/local/lib/python3.11/dist-packages/traitlets/config/application.py", line 1082, in launch_instance
    app.start()
  File "/usr/local/lib/python3.11/dist-packages/ipykernel/kernelapp.p

In [3]:
# ── Task mapping: survey question → Model C task label ─────────────────────
TASK_MAP = {
    "Overall, how accessible did you find the AAMI website?":                     "AAMI Website",
    "Generally speaking, how user friendly, inuitive, helpful was the AAMI website to use?": "AAMI Website",
    "How user-friendly was the AAMI Homepage?":                                   "AAMI Website",
    "How easy was it to find and read a product disclosure statement?":           "PDFs and Understanding Terms",
    "How easy and clear was the information on Natural disasters?":               "Equal Access to Critical  Information – Emergency Response Page",
    "How user-friendly was the search function?":                                 "Finding Answers",
    "How user friendly was the live chat?":                                       "Finding Answers",
    "How easy was it to navigate around the website?":                            "AAMI Website",
}

# ── L1.4 confidence threshold ──────────────────────────────────────────────
CONFIDENCE_THRESHOLD = 0.70

# ── Friction stream routing ────────────────────────────────────────────────
FRICTION_STREAM = {
    "accessibility":          "AT",
    "comprehension":          "COMP",
    "content_not_found":      "USA",
    "confidence":             "USA",
    "excessive_effort":       "USA",
    "unexpected_behaviour":   "USA",
    "unresponsive_interface": "USA",
}

# ── L2.7 cohort weights (acc_weight, usa_weight) for comprehension friction ─
COHORT_WEIGHTS = {
    "ESL": (1.0, 0.4),
    "DF":  (1.0, 0.4),
    "ND":  (1.0, 0.4),
    "BL":  (0.5, 0.9),
    "LV":  (0.5, 0.9),
    "OL":  (0.6, 0.8),
    "AB":  (0.4, 1.0),
    "OT":  (0.4, 1.0),
}
COHORT_PRIORITY       = ["ESL","DF","ND","BL","LV","OL","AB","OT"]
DEFAULT_COHORT_WEIGHT = (0.4, 1.0)

# ── Severity codes ─────────────────────────────────────────────────────────
SEV_BLOCKER = "blocker"
SEV_SEVERE  = "severe"
SEV_HIGH    = "high"
SEV_MEDIUM  = "medium"
SEV_LOW     = "low"

# ── Proposal 1 graduated deduction constants ───────────────────────────────
# (base_penalty, floor) — replaces hard caps for medium-severity friction
GRAD_MEDIUM = {
    "AT":       (20, 30),   # medium AT: 20-pt base penalty, floor 30
    "COMP":     (15, 50),   # medium COMP: 15-pt base penalty, floor 50
    "USA":      (12, 75),   # medium USA (single): 12-pt base penalty, floor 75
    "USA_3plus":(25, 70),   # 3+ medium USA: 25-pt base penalty, floor 70
}

print("Constants loaded.")

Constants loaded.


In [4]:
# Survey file is named .xls but is actually CSV
df_survey = pd.read_csv(SURVEY_PATH)
print(f"Survey rows: {len(df_survey)}, cols: {len(df_survey.columns)}")

EMAIL_COL         = "Please enter your email address"
TESTER_COL        = "What type of tester are you? (Select all that apply)"
MET_COL           = "How well did the AAMI  website meet your expectations?"
PROJECT_BLOCK_COL = (
    "Did you find you were blocked from completing some tasks "
    "because of poor functionality or because you didn't understand the content?"
)

assert EMAIL_COL in df_survey.columns, f"Missing column: {EMAIL_COL}"
assert MET_COL   in df_survey.columns, f"Missing column: {MET_COL}"
df_survey.head(2)

Survey rows: 21, cols: 55


,Timestamp,Please enter your email address,What type of tester are you? (Select all that apply),Select which assistive technologies/ adaoptions you used for this specific project,Preferred type of adaption/ assistive technology used for this specific project,"Overall, how accessible did you find the AAMI website?",Did you find you were blocked from completing some tasks because of poor functionality or because you didn't understand the content?,"Generally speaking, how user friendly, inuitive, helpful was the AAMI website to use?","Based on your experience today, do you think you would have been able to independently answer questions by searching information on this site?",Please select your age range,...,"Generally speaking, when using online services that require you to log in (for example, banking, utilities, health or government services), which of the following best describes how you usually prefer to manage the account?","Generally speaking, when using online services that require you to log in, which of the following best describes how you usually manage your passwords or sign-in details?","Now considering specific services, such as those provided by AAMI select how you'd prefer to manage your services & transactions","When using online services, would you generally prefer your personal information and service history to be saved so your future forms, services, and interactions can be faster and more personalised?",Select the state where you reside,Select the option that best describes your location,What's your postcode,Have you considered AAMI insurance before?,"Reflecting on the AAMI website, how did your experience compare with other services you use regularly?",What's the best insurance website you've used and describe how it was better than this experience
0,2026/02/06 9:23:51 pm GMT+11,reneerussell99@hotmail.com,Neurodivergent;Blind,Screen reader,VoiceOver,4,I could complete all tasks with workarounds,5,Very confident – I could easily find what I ne...,21-30,...,I use an account I created myself just for my ...,I reuse the same or similar passwords across m...,I would use an account I created myself just f...,I prefer my information to be saved so forms a...,Queensland,Metro location but not a capital city,4505,Yes,It was slightly easier to use than most other ...,AAMI has been the best website I’ve used for i...
1,2026/02/06 10:27:28 pm GMT+11,reececraigie@outlook.com,Aboriginal,Not applicable,not applicable,4,I could complete all tasks with ease,4,Somewhat confident – I could manage but might ...,31-40,...,I use an account I created myself just for my ...,I reuse the same or similar passwords across m...,I would use an account I created myself just f...,I prefer my information to be saved so forms a...,New South Wales,Regional,2340,Yes,It was slightly easier to use than most other ...,I use Allianz - I would say it comes down to b...


In [5]:
def normalise_met(text):
    if not isinstance(text, str):
        return "unknown"
    t = text.lower()
    if "exceeded" in t:                                  return "exceeded"
    if "fell well below" in t:                           return "fell_well_below"
    if "did not meet" in t or "didn't meet" in t:        return "not_met"
    if "partly" in t:                                    return "partly_met"
    if "met" in t:                                       return "met"
    return "unknown"

TESTER_TO_CODE = {
    "non-english speaking": "ESL", "esl": "ESL",
    "deaf": "DF", "hard of hearing": "DF",
    "neurodivergent": "ND",
    "blind": "BL",
    "low vision": "LV",
    "older": "OL", "70+": "OL",
    "aboriginal": "AB",
    "physically disabled": "OT", "others": "OT", "other": "OT",
}

def cohort_codes_from_label(text):
    if not isinstance(text, str):
        return []
    parts = [p.strip().lower() for p in text.replace(",", ";").split(";")]
    return [TESTER_TO_CODE[p] for p in parts if p in TESTER_TO_CODE]

def primary_cohort(codes):
    for c in COHORT_PRIORITY:
        if c in codes:
            return c
    return None

participants = pd.DataFrame({
    "email":            df_survey[EMAIL_COL].str.lower().str.strip(),
    "cohort_codes":     df_survey[TESTER_COL].apply(cohort_codes_from_label),
    "met_expectations": df_survey[MET_COL].apply(normalise_met),
    "project_block_self_report": df_survey[PROJECT_BLOCK_COL],
})
participants["primary_cohort"] = participants["cohort_codes"].apply(primary_cohort)
participants["acc_weight"] = participants["primary_cohort"].map(
    {k: v[0] for k, v in COHORT_WEIGHTS.items()}).fillna(DEFAULT_COHORT_WEIGHT[0])
participants["usa_weight"] = participants["primary_cohort"].map(
    {k: v[1] for k, v in COHORT_WEIGHTS.items()}).fillna(DEFAULT_COHORT_WEIGHT[1])
participants["project_blocked"] = participants["project_block_self_report"].fillna("").str.contains(
    "couldn't complete", case=False)

print(f"Participants: {len(participants)}")
participants[["email","primary_cohort","met_expectations","acc_weight","usa_weight","project_blocked"]].head()

Participants: 21


,email,primary_cohort,met_expectations,acc_weight,usa_weight,project_blocked
0,reneerussell99@hotmail.com,ND,exceeded,1.0,0.4,False
1,reececraigie@outlook.com,AB,met,0.4,1.0,False
2,giuliaclemente26@gmail.com,ESL,partly_met,1.0,0.4,False
3,mgblackwell2001@gmail.com,ND,not_met,1.0,0.4,False
4,carlpatrickrobinson@gmail.com,ND,exceeded,1.0,0.4,True


In [6]:
records = []
for surv_q, task in TASK_MAP.items():
    if surv_q not in df_survey.columns:
        continue
    sub = df_survey[[EMAIL_COL, surv_q, MET_COL]].copy()
    sub = sub.rename(columns={EMAIL_COL: "email", surv_q: "raw_score", MET_COL: "met_raw"})
    sub["task"]           = task
    sub["survey_question"] = surv_q
    records.append(sub)

rows = pd.concat(records, ignore_index=True)
rows["email"]         = rows["email"].str.lower().str.strip()
rows["raw_score"]     = pd.to_numeric(rows["raw_score"], errors="coerce")
rows["ceiling_score"] = rows["raw_score"] * 20.0   # 1-5 Likert → 0-100
rows = rows.dropna(subset=["ceiling_score"]).reset_index(drop=True)

rows = rows.merge(
    participants[["email","met_expectations","primary_cohort",
                  "acc_weight","usa_weight","project_blocked"]],
    on="email", how="left")

print(f"{len(rows)} (participant, survey_question) rows")
rows.head(3)

168 (participant, survey_question) rows


,email,raw_score,met_raw,task,survey_question,ceiling_score,met_expectations,primary_cohort,acc_weight,usa_weight,project_blocked
0,reneerussell99@hotmail.com,4,Exceeded my expectations — it was smoother and...,AAMI Website,"Overall, how accessible did you find the AAMI ...",80.0,exceeded,ND,1.0,0.4,False
1,reececraigie@outlook.com,4,Met my expectations,AAMI Website,"Overall, how accessible did you find the AAMI ...",80.0,met,AB,0.4,1.0,False
2,giuliaclemente26@gmail.com,3,Partly met my expectations — some areas worked...,AAMI Website,"Overall, how accessible did you find the AAMI ...",60.0,partly_met,ESL,1.0,0.4,False


In [7]:
df_insights = pd.read_excel(INSIGHTS_PATH)
df_insights["email_key"] = df_insights["tester_email"].str.lower().str.strip()
df_insights = df_insights[df_insights["category"].isin(["friction","blocker"])].copy()
df_insights["severity"]     = df_insights["severity"].fillna("").str.lower()
df_insights.loc[df_insights["category"]=="blocker","severity"] = SEV_BLOCKER
df_insights["friction_type"] = df_insights["friction_type"].fillna("")
df_insights["confidence"]    = pd.to_numeric(df_insights["confidence"], errors="coerce")
print(f"Insights loaded: {len(df_insights)} rows")
df_insights[["email_key","task_title","friction_type","severity","confidence"]].head(3)

Insights loaded: 572 rows


,email_key,task_title,friction_type,severity,confidence
3,terryaflint17@gmail.com,AAMI Website,comprehension,medium,0.82
4,terryaflint17@gmail.com,AAMI Website,content_not_found,medium,0.78
5,terryaflint17@gmail.com,AAMI Website,confidence,low,0.65


## Pre-processing — Semantic deduplication (L1 pre-check)

Insights with ≥ 85% cosine similarity on the same (participant, task) are merged.
The higher severity and higher confidence from each cluster are retained.
This prevents LLM over-generation from artificially inflating friction counts.

In [8]:
SEVERITY_ORDER = {"blocker":5,"blocked":5,"severe":4,"high":3,"medium":2,"low":1,"":0}
SIM_THRESHOLD  = 0.85

model_st = SentenceTransformer("all-MiniLM-L6-v2")
deduped_rows = []

for (email, task), group in df_insights.groupby(["email_key","task_title"]):
    group = group.copy()
    texts = (group["title"].fillna("") + " " + group["description"].fillna("")).tolist()
    if len(texts) <= 1:
        if len(group) == 1:
            deduped_rows.append(group.iloc[0].to_dict())
        continue
    embeddings  = model_st.encode(texts, convert_to_tensor=True)
    sim_matrix  = util.cos_sim(embeddings, embeddings)
    used = set()
    for i in range(len(group)):
        if i in used:
            continue
        duplicates = [i]
        for j in range(i+1, len(group)):
            if j not in used and sim_matrix[i][j] >= SIM_THRESHOLD:
                duplicates.append(j); used.add(j)
        merged   = group.iloc[duplicates].copy()
        best_idx = merged["severity"].map(SEVERITY_ORDER).idxmax()
        best_row = merged.loc[best_idx].copy()
        best_row["confidence"] = merged["confidence"].max()
        deduped_rows.append(best_row.to_dict())

df_insights_deduped = pd.DataFrame(deduped_rows)
print(f"Before: {len(df_insights):,} → after dedup: {len(df_insights_deduped):,}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Before: 572 → after dedup: 571


In [9]:
df_ins_trig = df_insights_deduped[
    df_insights_deduped["confidence"] >= CONFIDENCE_THRESHOLD].copy()
df_ins_trig["stream"] = df_ins_trig["friction_type"].map(FRICTION_STREAM)
print(f"Triggering insights (conf ≥ {CONFIDENCE_THRESHOLD}): {len(df_ins_trig):,}")
df_ins_trig["stream"].value_counts(dropna=False)

Triggering insights (conf ≥ 0.7): 528


stream
USA     320
COMP    143
AT       65
Name: count, dtype: int64

In [10]:
matched_emails = set(df_ins_trig["email_key"])
all_emails     = set(rows["email"])
unmatched      = sorted(all_emails - matched_emails)

if unmatched:
    print(f"Dropping {len(unmatched)} participants with no matched insights:")
    for e in unmatched:
        print(f"  - {e}")
    rows         = rows[rows["email"].isin(matched_emails)].reset_index(drop=True)
    participants = participants[participants["email"].isin(matched_emails)].reset_index(drop=True)
else:
    print("All participants have at least one matched insight.")

print(f"Remaining: {len(rows)} rows from {rows['email'].nunique()} participants")

Dropping 4 participants with no matched insights:
  - asmaosyan9@gmail.com
  - mattymc2@gmail.com
  - tdcn66@gmail.com
  - troyparnell@hotmail.com
Remaining: 136 rows from 17 participants


## Helper functions — friction counting, cap application, graduated deduction

In [11]:
def count_frictions(email, task, stream, severity):
    m = ((df_ins_trig["email_key"] == email) &
         (df_ins_trig["task_title"] == task) &
         (df_ins_trig["stream"]     == stream) &
         (df_ins_trig["severity"]   == severity))
    return int(m.sum())

def has_friction(email, task, stream, severity):
    return count_frictions(email, task, stream, severity) > 0

def max_confidence(email, task, stream, severity):
    m = ((df_ins_trig["email_key"] == email) &
         (df_ins_trig["task_title"] == task) &
         (df_ins_trig["stream"]     == stream) &
         (df_ins_trig["severity"]   == severity))
    vals = df_ins_trig.loc[m, "confidence"]
    return float(vals.max()) if len(vals) > 0 else 0.0

def apply_caps_with_weight(score, base_cap, weight):
    """Scale cap pressure by weight. weight=1.0 → full cap."""
    if score <= base_cap:
        return score
    return base_cap + (1.0 - weight) * (score - base_cap)

def apply_graduated_rule(current_score, base_penalty, floor, confidence):
    """Proposal 1: graduated deduction scales with confidence above 70%."""
    if confidence < CONFIDENCE_THRESHOLD or current_score <= floor:
        return current_score
    confidence_weight = (confidence - 0.70) / 0.30
    deduction = round(base_penalty * confidence_weight)
    return max(floor, current_score - deduction)

print("Helper functions defined.")

Helper functions defined.


## L1 — Per-row score (ceiling + AT hard caps)

- **L1.1** Survey score × 20 = ceiling (0–100)
- **L1.2** Blocked/severe AT → score = 0 (hard, not graduated)
- **L1.3** High AT: 1× → cap 20; 2+ → cap 10 (hard)
- **L1.4** Confidence threshold ≥ 0.70 to trigger any cap
- **L1.5** Confident-positive floor applied *after* L2/L3

In [12]:
def apply_l1(row):
    email, task = row["email"], row["task"]
    score = row["ceiling_score"]
    if has_friction(email, task, "AT", SEV_BLOCKER) or has_friction(email, task, "AT", SEV_SEVERE):
        return 0.0
    high_at = count_frictions(email, task, "AT", SEV_HIGH)
    if high_at >= 2:
        score = min(score, 10)
    elif high_at == 1:
        score = min(score, 20)
    return score

rows["l1_score"] = rows.apply(apply_l1, axis=1)
print("L1 applied.")
rows[["email","task","ceiling_score","l1_score"]].head(3)

L1 applied.


,email,task,ceiling_score,l1_score
0,reneerussell99@hotmail.com,AAMI Website,80.0,80.0
1,reececraigie@outlook.com,AAMI Website,80.0,80.0
2,giuliaclemente26@gmail.com,AAMI Website,60.0,60.0


## L2 — Accessibility score

- **L2.1** Medium AT → graduated deduction (base 20, floor 30)
- **L2.2** High/severe COMP → cap 40 (cohort-weighted)
- **L2.3** Medium COMP → graduated deduction (base 15, floor 50, cohort-weighted)
- **L2.4** 3+ medium COMP → hard cap 40 (cohort-weighted)
- **L2.5** 3+ low COMP → cap 60; any low COMP → cap 70 (cohort-weighted)
- **L2.6** Exceeded-expectations floor: comprehension stream cannot push ACC below 45
- **L2.7** Cohort-aware comprehension weights (ESL/DF/ND = 1.0×, BL/LV = 0.5×, OL = 0.6×, General = 0.4×)

> **Proposal 2 does not change Accessibility scores.** ACC scores produced here are final.

In [13]:
_comp_stacking_log = []

def apply_l2(row):
    email, task = row["email"], row["task"]
    score = row["l1_score"]
    if score == 0:
        return 0.0
    acc_w = row["acc_weight"]

    # L2.1 Medium AT — graduated
    n_med_at = count_frictions(email, task, "AT", SEV_MEDIUM)
    if n_med_at >= 1:
        conf  = max_confidence(email, task, "AT", SEV_MEDIUM)
        score = apply_graduated_rule(score, *GRAD_MEDIUM["AT"], conf)

    # Low AT count caps
    n_low_at = count_frictions(email, task, "AT", SEV_LOW)
    if n_low_at >= 3:
        score = min(score, 60)
    elif n_low_at >= 1:
        score = min(score, 70)

    # Track AT floor before comprehension
    at_floor = row["l1_score"]
    if n_med_at >= 1:
        conf     = max_confidence(email, task, "AT", SEV_MEDIUM)
        at_floor = apply_graduated_rule(at_floor, *GRAD_MEDIUM["AT"], conf)
    if n_low_at >= 3:
        at_floor = min(at_floor, 60)
    elif n_low_at >= 1:
        at_floor = min(at_floor, 70)

    # L2.2 High/severe COMP
    if has_friction(email, task, "COMP", SEV_SEVERE) or has_friction(email, task, "COMP", SEV_HIGH):
        score = apply_caps_with_weight(score, 40, acc_w)

    # L2.3/L2.4 Medium COMP
    n_med_comp = count_frictions(email, task, "COMP", SEV_MEDIUM)
    if n_med_comp >= 3:
        _comp_stacking_log.append({"email":email,"task":task,"stream":"ACC",
                                   "n_medium_comp":n_med_comp,"score_before":score,
                                   "met_expectations":row["met_expectations"]})
        score = apply_caps_with_weight(score, 40, acc_w)
    elif n_med_comp >= 1:
        conf      = max_confidence(email, task, "COMP", SEV_MEDIUM)
        graduated = apply_graduated_rule(score, *GRAD_MEDIUM["COMP"], conf)
        if graduated < score:
            score = apply_caps_with_weight(score, graduated, acc_w)

    # L2.5 Low COMP
    if count_frictions(email, task, "COMP", SEV_LOW) >= 3:
        score = apply_caps_with_weight(score, 60, acc_w)
    elif has_friction(email, task, "COMP", SEV_LOW):
        score = apply_caps_with_weight(score, 70, acc_w)

    # L2.6 Exceeded-expectations floor (comprehension stream only)
    if row["met_expectations"] == "exceeded":
        score = max(score, min(at_floor, 45))

    return score

rows["l2_score"] = rows.apply(apply_l2, axis=1)
print("L2 applied.")
rows[["email","task","l1_score","l2_score"]].head(3)

L2 applied.


,email,task,l1_score,l2_score
0,reneerussell99@hotmail.com,AAMI Website,80.0,80.00
1,reececraigie@outlook.com,AAMI Website,80.0,74.08
2,giuliaclemente26@gmail.com,AAMI Website,60.0,51.00


## L3 — Usability score

- **L3.1** Severe USA or 3+ high USA → cap 50
- **L3.2** High + medium USA → cap 60
- **L3.3** Medium + low USA → graduated (base 12, floor 75)
- **L3.4** 3+ medium USA → graduated (base 25, floor 70)
- **L3.5** 3+ low USA → cap 80
- **L3.6** Most restrictive cap wins
- **L3.7** Met-expectations moderator: not_met × 0.80; fell_well_below × 0.60
- **L3.8** Time moderation (Proposal 2) — applied in Notebook 02, **not here**

> **Proposal 2 only changes L3.8 time moderation. It does not change Accessibility scores.**

In [14]:
def apply_l3(row):
    email, task = row["email"], row["task"]
    score = row["l1_score"]
    if score == 0:
        return 0.0
    usa_w = row["usa_weight"]

    n_severe = count_frictions(email, task, "USA", SEV_SEVERE)
    n_high   = count_frictions(email, task, "USA", SEV_HIGH)
    n_med    = count_frictions(email, task, "USA", SEV_MEDIUM)
    n_low    = count_frictions(email, task, "USA", SEV_LOW)

    candidates = []
    if n_severe >= 1 or n_high >= 3:
        candidates.append(min(score, 50))
    if n_high >= 1 and n_med >= 1:
        candidates.append(min(score, 60))
    if n_med >= 3:
        conf = max_confidence(email, task, "USA", SEV_MEDIUM)
        candidates.append(apply_graduated_rule(score, *GRAD_MEDIUM["USA_3plus"], conf))
    if n_med >= 1 and n_low >= 1:
        conf = max(max_confidence(email, task, "USA", SEV_MEDIUM),
                   max_confidence(email, task, "USA", SEV_LOW))
        candidates.append(apply_graduated_rule(score, *GRAD_MEDIUM["USA"], conf))
    if n_low >= 3:
        candidates.append(min(score, 80))
    if candidates:
        score = min(candidates)

    # Comprehension in USA stream (L2.7 usa_weight)
    if has_friction(email, task, "COMP", SEV_SEVERE) or has_friction(email, task, "COMP", SEV_HIGH):
        score = apply_caps_with_weight(score, 40, usa_w)
    n_med_comp = count_frictions(email, task, "COMP", SEV_MEDIUM)
    if n_med_comp >= 3:
        _comp_stacking_log.append({"email":email,"task":task,"stream":"USA",
                                   "n_medium_comp":n_med_comp,"score_before":score,
                                   "met_expectations":row["met_expectations"]})
        score = apply_caps_with_weight(score, 40, usa_w)
    elif n_med_comp >= 1:
        conf      = max_confidence(email, task, "COMP", SEV_MEDIUM)
        graduated = apply_graduated_rule(score, *GRAD_MEDIUM["COMP"], conf)
        if graduated < score:
            score = apply_caps_with_weight(score, graduated, usa_w)
    if count_frictions(email, task, "COMP", SEV_LOW) >= 3:
        score = apply_caps_with_weight(score, 60, usa_w)
    elif has_friction(email, task, "COMP", SEV_LOW):
        score = apply_caps_with_weight(score, 70, usa_w)

    # L3.7 Met-expectations moderator
    me = row["met_expectations"]
    if me == "not_met":
        score *= 0.80
    elif me == "fell_well_below":
        score *= 0.60

    return score

rows["l3_score"] = rows.apply(apply_l3, axis=1)
print("L3 applied.")
rows[["email","task","l1_score","l2_score","l3_score"]].head(3)

L3 applied.


,email,task,l1_score,l2_score,l3_score
0,reneerussell99@hotmail.com,AAMI Website,80.0,80.00,80.0
1,reececraigie@outlook.com,AAMI Website,80.0,74.08,67.0
2,giuliaclemente26@gmail.com,AAMI Website,60.0,51.00,56.4


## L1.5 — Confident-positive floor

Applied **after** L2 and L3. If survey score ≥ 75, met_expectations is "met" or "exceeded",
and no high/severe frictions were identified, the final score cannot fall below 55.

In [15]:
_l15_override_log = []

def confident_positive_floor(row, score, stream_label=""):
    if score == 0:
        return 0.0
    cond_score = row["ceiling_score"] >= 75
    cond_met   = row["met_expectations"] in {"met","exceeded"}
    email, task = row["email"], row["task"]
    cond_no_high = (
        not has_friction(email, task, "AT",   SEV_BLOCKER) and
        not has_friction(email, task, "AT",   SEV_SEVERE)  and
        not has_friction(email, task, "AT",   SEV_HIGH)    and
        not has_friction(email, task, "USA",  SEV_SEVERE)  and
        not has_friction(email, task, "USA",  SEV_HIGH)    and
        not has_friction(email, task, "COMP", SEV_SEVERE)  and
        not has_friction(email, task, "COMP", SEV_HIGH)
    )
    if cond_score and cond_met and cond_no_high:
        restored = max(score, 55)
        if restored > score:
            _l15_override_log.append({"email":email,"task":task,"stream":stream_label,
                                      "score_before_l15":score,"score_after_l15":restored})
        return restored
    return score

rows["acc_score_final"] = rows.apply(lambda r: confident_positive_floor(r, r["l2_score"], "ACC"), axis=1)
rows["usa_score_final"] = rows.apply(lambda r: confident_positive_floor(r, r["l3_score"], "USA"), axis=1)

total = len(rows)
n_overrides = len(_l15_override_log)
print(f"L1.5 overrides: {n_overrides} / {total} ({100*n_overrides/total:.1f}%)")
if n_overrides / total > 0.05:
    print("WARNING: L1.5 override rate >5% — consider reviewing base penalties.")

L1.5 overrides: 0 / 136 (0.0%)


## Score explanations

In [16]:
def explain_score(row, stream="ACC"):
    email, task = row["email"], row["task"]
    ceiling = int(row["ceiling_score"])
    l1      = int(row["l1_score"])
    final   = int(row["acc_score_final"] if stream=="ACC" else row["usa_score_final"])
    parts   = []

    if stream == "ACC":
        n_med_at = count_frictions(email, task, "AT", SEV_MEDIUM)
        if n_med_at >= 1:
            conf = max_confidence(email, task, "AT", SEV_MEDIUM)
            cw   = (conf - 0.70) / 0.30
            ded  = round(20 * cw)
            barrier = "barriers" if n_med_at > 1 else "barrier"
            parts.append(f"{n_med_at} medium accessibility {barrier} (−{ded} pts at {conf:.0%} confidence, floor 30)")
        n_med_comp = count_frictions(email, task, "COMP", SEV_MEDIUM)
        if n_med_comp >= 3:
            parts.append(f"{n_med_comp} medium comprehension barriers (hard cap 40 — stacking)")
        elif n_med_comp >= 1:
            conf  = max_confidence(email, task, "COMP", SEV_MEDIUM)
            cw    = (conf - 0.70) / 0.30
            ded   = round(15 * cw)
            acc_w = row["acc_weight"]
            wn    = f", cohort weight {acc_w:.1f}×" if acc_w != 1.0 else ""
            barrier = "barriers" if n_med_comp > 1 else "barrier"
            parts.append(f"{n_med_comp} medium comprehension {barrier} (−{ded} pts at {conf:.0%} confidence, floor 50{wn})")
    else:
        n_severe = count_frictions(email, task, "USA", SEV_SEVERE)
        n_high   = count_frictions(email, task, "USA", SEV_HIGH)
        n_med    = count_frictions(email, task, "USA", SEV_MEDIUM)
        n_low    = count_frictions(email, task, "USA", SEV_LOW)
        if n_med >= 3:
            conf = max_confidence(email, task, "USA", SEV_MEDIUM)
            cw   = (conf - 0.70) / 0.30
            ded  = round(25 * cw)
            parts.append(f"{n_med} medium usability barriers (stacking: −{ded} pts at {conf:.0%} confidence, floor 70)")
        elif n_med >= 1 and n_low >= 1:
            conf = max(max_confidence(email, task, "USA", SEV_MEDIUM),
                       max_confidence(email, task, "USA", SEV_LOW))
            cw   = (conf - 0.70) / 0.30
            ded  = round(12 * cw)
            parts.append(f"{n_med} medium + {n_low} low usability barriers (−{ded} pts at {conf:.0%} confidence, floor 75)")
        n_med_comp = count_frictions(email, task, "COMP", SEV_MEDIUM)
        if n_med_comp >= 3:
            parts.append(f"{n_med_comp} medium comprehension barriers (hard cap 40 — stacking)")
        elif n_med_comp >= 1:
            conf  = max_confidence(email, task, "COMP", SEV_MEDIUM)
            cw    = (conf - 0.70) / 0.30
            ded   = round(15 * cw)
            usa_w = row["usa_weight"]
            wn    = f", cohort weight {usa_w:.1f}×" if usa_w != 1.0 else ""
            barrier = "barriers" if n_med_comp > 1 else "barrier"
            parts.append(f"{n_med_comp} medium comprehension {barrier} (−{ded} pts at {conf:.0%} confidence, floor 50{wn})")
        if n_severe >= 1 or n_high >= 3:
            parts.append("severe/3+ high usability friction (hard cap 50)")
        elif n_high >= 1 and n_med >= 1:
            parts.append("high + medium usability friction (hard cap 60)")

    l15_fired = any(e["email"]==email and e["task"]==task and e["stream"]==stream for e in _l15_override_log)

    if l1 == 0:
        return (f"This participant was categorically blocked on this task "
                f"(survey score {ceiling}). Their {stream} score is 0.")
    if not parts:
        if ceiling == final:
            return (f"No friction rules changed this participant's {stream} score. "
                    f"Final score: {final} (from survey score {ceiling}).")
        return (f"This participant's {stream} score of {final} reflects a hard AT cap "
                f"applied at L1 (survey score {ceiling}). No medium friction rules fired.")
    friction_str = "; ".join(parts)
    floor_note   = f" The L1.5 confident-positive floor restored the score to {final}." if l15_fired else ""
    if ceiling == final:
        return (f"Friction was identified ({friction_str}) but did not reduce "
                f"this participant's {stream} score below their survey score of {ceiling}. "
                f"Final score: {final}.")
    return (f"This participant's {stream} score of {final} reflects a reduction "
            f"from their survey score of {ceiling} due to: {friction_str}.{floor_note}")

rows["acc_explanation"] = rows.apply(lambda r: explain_score(r, "ACC"), axis=1)
rows["usa_explanation"] = rows.apply(lambda r: explain_score(r, "USA"), axis=1)
print("Explanations generated.")
rows[["email","task","ceiling_score","acc_score_final","acc_explanation"]].head(3)

Explanations generated.


,email,task,ceiling_score,acc_score_final,acc_explanation
0,reneerussell99@hotmail.com,AAMI Website,80.0,80.00,No friction rules changed this participant's A...
1,reececraigie@outlook.com,AAMI Website,80.0,74.08,This participant's ACC score of 74 reflects a ...
2,giuliaclemente26@gmail.com,AAMI Website,60.0,51.00,This participant's ACC score of 51 reflects a ...


## Rollups — task, cohort, project

In [17]:
def _stat(g, col):
    mask = g["l1_score"] > 0
    if mask.sum() == 0:
        return float("nan"), float("nan")
    s = g.loc[mask, col]
    return s.mean(), s.std()

task_summary = (
    rows.groupby("task")
    .apply(lambda g: pd.Series({
        "n_rows":      len(g),
        "n_blocked":   int((g["l1_score"]==0).sum()),
        "task_acc":    _stat(g,"acc_score_final")[0],
        "task_acc_sd": _stat(g,"acc_score_final")[1],
        "task_usa":    _stat(g,"usa_score_final")[0],
        "task_usa_sd": _stat(g,"usa_score_final")[1],
    }))
    .reset_index()
)
print("Task summary:")
print(task_summary.to_string(index=False))

Task summary:
                                                           task  n_rows  n_blocked  task_acc  task_acc_sd  task_usa  task_usa_sd
                                                   AAMI Website    68.0        0.0 74.382647    20.579623 74.748882    21.251886
Equal Access to Critical  Information – Emergency Response Page    17.0        0.0 81.400000    15.999844 77.276471    15.948649
                                                Finding Answers    34.0        0.0 41.779412    28.725237 38.233529    24.883212
                                   PDFs and Understanding Terms    17.0        2.0 60.586667    21.971308 56.821333    22.319491


In [18]:
pt = (rows.groupby(["email","primary_cohort","task"], dropna=False)
          [["acc_score_final","usa_score_final"]]
          .mean().reset_index())

participant_summary = (
    pt.groupby(["email","primary_cohort"], dropna=False)
      [["acc_score_final","usa_score_final"]]
      .mean().reset_index()
      .rename(columns={"acc_score_final":"participant_acc","usa_score_final":"participant_usa"})
)

cohort_summary = (
    participant_summary.groupby("primary_cohort", dropna=False)
    .agg(n=("email","nunique"),
         cohort_acc=("participant_acc","mean"),
         cohort_acc_sd=("participant_acc","std"),
         cohort_usa=("participant_usa","mean"),
         cohort_usa_sd=("participant_usa","std"))
    .reset_index()
)
print("Cohort summary:")
print(cohort_summary.to_string(index=False))

Cohort summary:
primary_cohort  n  cohort_acc  cohort_acc_sd  cohort_usa  cohort_usa_sd
            AB  1   65.520000            NaN   63.000000            NaN
            BL  2   60.468750      23.820660   59.343750      14.239363
            DF  2   67.500000      10.606602   70.475000      17.005918
           ESL  1   55.125000            NaN   56.280000            NaN
            LV  4   54.945312      24.434255   51.314062      19.259707
            ND  4   62.000000       4.928658   60.028500       8.809558
            OL  2   73.262500       7.194811   65.071250       8.808783
            OT  1   75.950000            NaN   67.250000            NaN


In [19]:
participant_summary = participant_summary.merge(
    participants[["email","project_blocked"]], on="email", how="left")

participant_summary["project_acc"] = np.where(
    participant_summary["project_blocked"].fillna(False), 0.0,
    participant_summary["participant_acc"])

project_acc    = participant_summary["project_acc"].mean()
project_acc_sd = participant_summary["project_acc"].std()
project_usa    = participant_summary["participant_usa"].mean()
project_usa_sd = participant_summary["participant_usa"].std()

print(f"Project accessibility:  {project_acc:6.2f}  (sd {project_acc_sd:.2f})")
print(f"Project usability:      {project_usa:6.2f}  (sd {project_usa_sd:.2f})")

Project accessibility:   50.72  (sd 31.47)
Project usability:       60.10  (sd 12.66)


## Save outputs

In [20]:
# Export proposal1_rows.csv (used by Notebook 02)
rows.to_csv(OUTPUT_DIR / "proposal1_rows.csv", index=False)

# Export proposal1_df_insights.csv (used by Notebook 02 and 04)
df_insights_deduped.to_csv(OUTPUT_DIR / "proposal1_df_insights.csv", index=False)

# Export task / cohort / project summaries
task_summary.to_csv(OUTPUT_DIR / "proposal1_task_summary.csv", index=False)
cohort_summary.to_csv(OUTPUT_DIR / "proposal1_cohort_summary.csv", index=False)

project_txt = (
    f"Project accessibility: {project_acc:.2f} (sd {project_acc_sd:.2f})\n"
    f"Project usability:     {project_usa:.2f} (sd {project_usa_sd:.2f})\n"
)
(OUTPUT_DIR / "proposal1_project_summary.txt").write_text(project_txt)

# Export full results Excel
xlsx_path = OUTPUT_DIR / "model_c3_proposal1_results.xlsx"
with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    rows.to_excel(writer, sheet_name="Scores", index=False)
    if _comp_stacking_log:
        pd.DataFrame(_comp_stacking_log).to_excel(writer, sheet_name="Comp_Stacking_Log", index=False)
    if _l15_override_log:
        pd.DataFrame(_l15_override_log).to_excel(writer, sheet_name="L15_Override_Log", index=False)

print(f"Saved {len(rows)} rows → {xlsx_path}")
print(f"Comp stacking events: {len(_comp_stacking_log)}")
print(f"L1.5 overrides:       {len(_l15_override_log)}")
print()
print("Output files:")
for f in sorted(OUTPUT_DIR.glob("proposal1_*")) + [xlsx_path]:
    print(f"  {f.name}")

Saved 136 rows → /home/ubuntu/cs19_capstone/outputs/model_c3_proposal1_results.xlsx
Comp stacking events: 0
L1.5 overrides:       0

Output files:
  proposal1_cohort_summary.csv
  proposal1_df_insights.csv
  proposal1_project_summary.txt
  proposal1_rows.csv
  proposal1_task_summary.csv
  model_c3_proposal1_results.xlsx


In [21]:
print("Sample explanations (rows with score reduction):")
for _, r in rows[rows["acc_explanation"].str.contains("reduction", na=False)].head(3).iterrows():
    print(f"  [{r['email'][:25]} | {r['task'][:30]}]")
    print(f"  ACC: {r['acc_explanation']}")
    print(f"  USA: {r['usa_explanation']}")
    print()

Sample explanations (rows with score reduction):
  [reececraigie@outlook.com | AAMI Website]
  ACC: This participant's ACC score of 74 reflects a reduction from their survey score of 80 due to: 1 medium comprehension barrier (−8 pts at 85% confidence, floor 50, cohort weight 0.4×).
  USA: This participant's USA score of 67 reflects a reduction from their survey score of 80 due to: 1 medium + 1 low usability barriers (−7 pts at 88% confidence, floor 75); 1 medium comprehension barrier (−8 pts at 85% confidence, floor 50).

  [giuliaclemente26@gmail.co | AAMI Website]
  ACC: This participant's ACC score of 51 reflects a reduction from their survey score of 60 due to: 2 medium comprehension barriers (−9 pts at 88% confidence, floor 50).
  USA: This participant's USA score of 56 reflects a reduction from their survey score of 60 due to: 2 medium comprehension barriers (−9 pts at 88% confidence, floor 50, cohort weight 0.4×).

  [ghum@me.com | AAMI Website]
  ACC: This participant's ACC sco